# Prerequisites: Creating Sample Agents

## Overview

Let's first start by creating agents to be evaluated. This tutorial creates two sample agents for evaluation using different frameworks:
- [Strands Agents SDK](https://strandsagents.com/)
- [LangGraph](https://www.langchain.com/langgraph)

Both agents uses Anthropic Claude Haiku 4.5 from Amazon Bedrock as the LLM model but you can use any model of your preference and they have identical capabilities:
- **Math Tool**: Tool to perform basic math operations
- **Weather Tool**: Dummy implementation for weather tool


The architecture looks as following:

![Architecture](../images/agent_architecture.png)

## Prerequisites
- Python 3.10+
- AWS credentials

In [1]:
!pip install -r ../requirements.txt

INFO: pip is looking at multiple versions of bedrock-agentcore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of bedrock-agentcore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 1.2 MB/s eta 0:00:02
   ---------------------------- ----------- 1.6/2.2 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 2.8 MB/s  0:00:01
   ---------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

## Setup

Import required packages and configure AWS session:

In [1]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import uuid
import time

boto_session = Session()
region = boto_session.region_name
print(f"Using region: {region}")

Using region: None


## Deploy Strands Agent
Let's deploy our Strands Agents to AgentCore Runtime

In [23]:
import os
os.environ["AWS_ACCESS_KEY_ID"] = "<your-access-key-id>"
os.environ["AWS_SECRET_ACCESS_KEY"] = "<your-secret-access-key>"
#os.environ["AWS_SESSION_TOKEN"] = "<your-session-token>"  # only if using temporary credentials
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"


In [24]:
import boto3

# Reset boto3 session to pick up new env vars
boto3.DEFAULT_SESSION = None

sts = boto3.client("sts")
print(sts.get_caller_identity())


{'UserId': 'AIDAQ2FIGGE7VURN6J7XQ', 'Account': '056187302207', 'Arn': 'arn:aws:iam::056187302207:user/adeline-pepela', 'ResponseMetadata': {'RequestId': 'fa0280e6-11de-45b0-812c-79129a1da437', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'fa0280e6-11de-45b0-812c-79129a1da437', 'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTE6UzoxNzg3NzIzOTc1OTgxOlI6ZEs1bDBJTFU=', 'content-type': 'text/xml', 'content-length': '411', 'date': 'Wed, 26 Aug 2026 05:59:35 GMT'}, 'RetryAttempts': 0}}


### Check status for deployment on AgentCore Runtime
Wait for deployment to be in ACTIVE status..

In [26]:
def wait_for_deployment(runtime, name):
    end_statuses = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
    while True:
        status_response = runtime.status()
        status = status_response.endpoint["status"]
        print(f"{name} status: {status}")
        if status in end_statuses:
            print(f"{name} deployment completed with status: {status}")
            return status
        time.sleep(10)


strands_status = wait_for_deployment(agentcore_runtime, "Strands")

Retrieved Bedrock AgentCore status for: ac_eval_strands2


Strands status: READY
Strands deployment completed with status: READY


### Invoke the Strands Agents on Runtime
Let's test the Strands agent by invoking the AgentCore Runtime endpoint with a payload.

In [27]:
session_id_strands = str(uuid.uuid4())
print(f"Session ID: {session_id_strands}")

Session ID: 49f812fe-a8f4-4891-8fa8-a1722672e3ef


In [35]:
import boto3

bedrock = boto3.client("bedrock", region_name=region)
models = bedrock.list_foundation_models(byProvider="Anthropic")
for m in models["modelSummaries"]:
    print(m["modelId"])


anthropic.claude-sonnet-4-20250514-v1:0
anthropic.claude-haiku-4-5-20251001-v1:0
anthropic.claude-fable-5
anthropic.claude-sonnet-4-6
anthropic.claude-opus-4-6-v1
anthropic.claude-opus-5
anthropic.claude-opus-4-8
anthropic.claude-opus-4-7
anthropic.claude-sonnet-4-5-20250929-v1:0
anthropic.claude-sonnet-5
anthropic.claude-opus-4-1-20250805-v1:0
anthropic.claude-opus-4-5-20251101-v1:0
anthropic.claude-3-haiku-20240307-v1:0:48k
anthropic.claude-3-haiku-20240307-v1:0:200k
anthropic.claude-3-haiku-20240307-v1:0


In [38]:
agentcore_runtime = Runtime()
response = agentcore_runtime.configure(
    entrypoint="eval_agent_strands.py",
    execution_role="arn:aws:iam::056187302207:role/role_agentcore",
    auto_create_execution_role=False,
    auto_create_ecr=True,
    requirements_file="requirements_strands.txt",
    region=region,
    agent_name=agent_name,
    idle_timeout=120,
)
agentcore_runtime.launch()


Entrypoint parsed: file=C:\Users\user\Downloads\ACD_Accra_Ghana\Accra_2026\agentcore-samples\06-workshops\07-AgentCore-evaluations\00-prereqs\eval_agent_strands.py, bedrock_agentcore_name=eval_agent_strands
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: ac_eval_strands2


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

📄 Using existing Dockerfile: 
c:\Users\user\Downloads\ACD_Accra_Ghana\Accra_2026\agentcore-samples\06-workshops\07-AgentCore-evaluations\00-prere
qs\Dockerfile

Generated .dockerignore: c:\Users\user\Downloads\ACD_Accra_Ghana\Accra_2026\agentcore-samples\06-workshops\07-AgentCore-evaluations\00-prereqs\.dockerignore
Keeping 'ac_eval_strands2' as default agent
Bedrock AgentCore configured: c:\Users\user\Downloads\ACD_Accra_Ghana\Accra_2026\agentcore-samples\06-workshops\07-AgentCore-evaluations\00-prereqs\.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'ac_eval_strands2' to account 056187302207 (us-east-1)
Generated image tag: 20260826-061127-814
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: ac_eval_strands2
EC

✅ Reusing existing ECR repository: 056187302207.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-ac_eval_strands2


Getting or creating CodeBuild execution role for agent: ac_eval_strands2
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-a67917b625
Reusing existing CodeBuild execution role: arn:aws:iam::056187302207:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-a67917b625
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: ac_eval_strands2/source.zip
Updated CodeBuild project: bedrock-agentcore-ac_eval_strands2-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.3s
🔄 PROVISIONING started (total: 2s)
✅ PROVISIONING completed in 5.1s
🔄 DOWNLOAD_SOURCE started (total: 7s)
✅ DOWNLOAD_SOURCE completed in 1.3s
🔄 BUILD started (total: 8s)
✅ BUILD completed in 16.7s
🔄 POST_BUILD started (total: 25s)
✅ POST_BUILD completed in 11.6s
🔄 COMPLETED started (total: 36s)
✅ COMPLETED completed in 1.3s
🎉 CodeBuild completed successfully in 0m 37s
CodeBuild completed succes

LaunchResult(mode='codebuild', tag='bedrock_agentcore-ac_eval_strands2:None', env_vars=None, port=None, runtime=None, ecr_uri='056187302207.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-ac_eval_strands2:20260826-061127-814', agent_id='ac_eval_strands2-m5noRdAhvA', agent_arn='arn:aws:bedrock-agentcore:us-east-1:056187302207:runtime/ac_eval_strands2-m5noRdAhvA', codebuild_id='bedrock-agentcore-ac_eval_strands2-builder:a1c8c253-8852-4892-8519-0604b4b23f7e', build_output=None)

In [42]:
import boto3

logs = boto3.client("logs", region_name=region)

# List all log groups with agentcore in the name
paginator = logs.get_paginator("describe_log_groups")
for page in paginator.paginate(logGroupNamePrefix="/aws/bedrock"):
    for lg in page["logGroups"]:
        print(lg["logGroupName"])



/aws/bedrock-agentcore/runtimes/ac_eval_strands2-m5noRdAhvA-DEFAULT


In [44]:
import boto3

# Check correct boto3 client methods
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)
print([m for m in dir(agentcore_client) if not m.startswith("_")])


['batch_create_memory_records', 'batch_delete_memory_records', 'batch_update_memory_records', 'can_paginate', 'close', 'complete_resource_token_auth', 'create_event', 'delete_event', 'delete_memory_record', 'evaluate', 'exceptions', 'generate_presigned_url', 'get_agent_card', 'get_browser_session', 'get_code_interpreter_session', 'get_event', 'get_memory_record', 'get_paginator', 'get_resource_api_key', 'get_resource_oauth2_token', 'get_waiter', 'get_workload_access_token', 'get_workload_access_token_for_jwt', 'get_workload_access_token_for_user_id', 'invoke_agent_runtime', 'invoke_agent_runtime_command', 'invoke_browser', 'invoke_code_interpreter', 'list_actors', 'list_browser_sessions', 'list_code_interpreter_sessions', 'list_events', 'list_memory_extraction_jobs', 'list_memory_records', 'list_sessions', 'meta', 'retrieve_memory_records', 'save_browser_session_profile', 'search_registry_records', 'start_browser_session', 'start_code_interpreter_session', 'start_memory_extraction_job'

In [45]:
# Check all bedrock-related log groups
logs = boto3.client("logs", region_name=region)

paginator = logs.get_paginator("describe_log_groups")
for page in paginator.paginate():
    for lg in page["logGroups"]:
        if "bedrock" in lg["logGroupName"].lower() or "agentcore" in lg["logGroupName"].lower():
            print(lg["logGroupName"])


/aws/bedrock-agentcore/runtimes/ac_eval_strands2-m5noRdAhvA-DEFAULT
/aws/codebuild/bedrock-agentcore-ac_eval_strands2-builder


In [46]:
import boto3

logs = boto3.client("logs", region_name=region)

log_group = "/aws/bedrock-agentcore/runtimes/ac_eval_strands2-m5noRdAhvA-DEFAULT"

streams = logs.describe_log_streams(
    logGroupName=log_group,
    orderBy="LastEventTime",
    descending=True,
    limit=3
)

stream_name = streams["logStreams"][0]["logStreamName"]
print("Stream:", stream_name)

events = logs.get_log_events(
    logGroupName=log_group,
    logStreamName=stream_name,
    limit=50
)

for e in events["events"]:
    print(e["message"])


Stream: 2026/08/26/[runtime-logs]a34b81fa-b62c-4506-9196-8abecdca3684
User input: How much is 2+2?
{"timestamp": "2026-08-26T06:12:52.267Z", "level": "ERROR", "message": "Invocation failed (0.205s)", "logger": "bedrock_agentcore.app", "requestId": "b2e664b2-04ac-423f-9dd3-e84bc72c6d48", "sessionId": "49f812fe-a8f4-4891-8fa8-a1722672e3ef", "errorType": "ResourceNotFoundException", "errorMessage": "An error occurred (ResourceNotFoundException) when calling the ConverseStream operation: Model use case details have not been submitted for this account. Fill out the Anthropic use case details form before using the model. If you have already filled out the form, try again in 15 minutes.", "stackTrace": ["Traceback (most recent call last):\n", "  File \"/usr/local/lib/python3.14/site-packages/bedrock_agentcore/runtime/app.py\", line 576, in _handle_invocation\n    result = await self._invoke_handler(handler, request_context, takes_context, payload)\n             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [47]:
build_log_group = "/aws/codebuild/bedrock-agentcore-ac_eval_strands2-builder"

streams = logs.describe_log_streams(
    logGroupName=build_log_group,
    orderBy="LastEventTime",
    descending=True,
    limit=3
)

stream_name = streams["logStreams"][0]["logStreamName"]
events = logs.get_log_events(
    logGroupName=build_log_group,
    logStreamName=stream_name,
    limit=50
)

for e in events["events"]:
    print(e["message"])


Pushing versioned image to ECR...



[Container] 2026/08/26 06:12:00.252788 Running command docker push 056187302207.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-ac_eval_strands2:20260826-061127-814

The push refers to repository [056187302207.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-ac_eval_strands2]

e086db01ab40: Preparing

e9e12d6ba895: Preparing

3bcd90fa9fac: Preparing

2035a2024ba8: Preparing

2c87cb00bace: Preparing

4f1ee0dcc6c7: Preparing

cb66fa58c1b1: Preparing

db772ae88d0b: Preparing

b348c9d47a0a: Preparing

e3e375a22f11: Preparing

bd390c400455: Preparing

cb66fa58c1b1: Waiting

db772ae88d0b: Waiting

b348c9d47a0a: Waiting

e3e375a22f11: Waiting

bd390c400455: Waiting

4f1ee0dcc6c7: Waiting

e086db01ab40: Pushed

e9e12d6ba895: Pushed

2c87cb00bace: Pushed

db772ae88d0b: Layer already exists

b348c9d47a0a: Layer already exists

e3e375a22f11: Layer already exists

3bcd90fa9fac: Pushed

4f1ee0dcc6c7: Pushed

cb66fa58c1b1: Pushed

bd390c400455: Pushed

2035a20

In [50]:
import boto3

bedrock = boto3.client("bedrock-runtime", region_name=region)

try:
    response = bedrock.converse(
        modelId="anthropic.claude-haiku-4-5-20251001-v1:0",
        messages=[{"role": "user", "content": [{"text": "hi"}]}]
    )
    print("Model accessible!")
except Exception as e:
    print(str(e))


An error occurred (ValidationException) when calling the Converse operation: Invocation of model ID anthropic.claude-haiku-4-5-20251001-v1:0 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.


In [51]:
invoke_response = agentcore_runtime.invoke(payload={"prompt": "How much is 2+2?"}, session_id=session_id_strands)
invoke_response

{'ResponseMetadata': {'RequestId': 'c3d0d29c-9fd7-45bb-91f2-e82c2fd38aed',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 26 Aug 2026 06:28:34 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'c3d0d29c-9fd7-45bb-91f2-e82c2fd38aed',
   'x-amzn-bedrock-agentcore-runtime-session-id': '49f812fe-a8f4-4891-8fa8-a1722672e3ef'},
  'RetryAttempts': 0},
 'runtimeSessionId': '49f812fe-a8f4-4891-8fa8-a1722672e3ef',
 'contentType': 'application/json',
 'statusCode': 200,
 'response': ['2 + 2 = **4**']}

In [52]:
invoke_response = agentcore_runtime.invoke(payload={"prompt": "How is the weather now?"}, session_id=session_id_strands)
invoke_response

{'ResponseMetadata': {'RequestId': '4987dbf5-306f-49a5-8d40-1ed4b509816b',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 26 Aug 2026 06:28:51 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': '4987dbf5-306f-49a5-8d40-1ed4b509816b',
   'x-amzn-bedrock-agentcore-runtime-session-id': '49f812fe-a8f4-4891-8fa8-a1722672e3ef'},
  'RetryAttempts': 0},
 'runtimeSessionId': '49f812fe-a8f4-4891-8fa8-a1722672e3ef',
 'contentType': 'application/json',
 'statusCode': 200,
 'response': ['The weather is currently **sunny**! 🌞']}

In [53]:
invoke_response = agentcore_runtime.invoke(
    payload={"prompt": "Can you tell me the capital of the US?"},
    session_id=session_id_strands,
)
invoke_response

{'ResponseMetadata': {'RequestId': 'ed96527a-1d01-4ce6-8989-807a3e37634f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 26 Aug 2026 06:29:02 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'ed96527a-1d01-4ce6-8989-807a3e37634f',
   'x-amzn-bedrock-agentcore-runtime-session-id': '49f812fe-a8f4-4891-8fa8-a1722672e3ef'},
  'RetryAttempts': 0},
 'runtimeSessionId': '49f812fe-a8f4-4891-8fa8-a1722672e3ef',
 'contentType': 'application/json',
 'statusCode': 200,
 'response': ["The capital of the United States is **Washington, D.C.** (District of Columbia).\n\nWashington, D.C. is located on the east coast and has been the capital since 1800. It's home to the White House (the residence and workplace of the President), the Capitol Building (where Congress meets), and the Supreme Court, among many other important government buildings and monuments."]}

## Deploy LangGraph Agent to AgentCore Runtime

Let's also deploy our LangGraph agent to AgentCore Runtime.

In [54]:
langgraph_agentcore_runtime = Runtime()
agent_name = "ac_eval_langgraph2"
response = langgraph_agentcore_runtime.configure(
    entrypoint="eval_agent_langgraph.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements_langgraph.txt",
    region=region,
    agent_name=agent_name,
    idle_timeout=120,
)
launch_result_langgraph = langgraph_agentcore_runtime.launch()
print(f"LangGraph agent deployment started: {launch_result_langgraph}")

Entrypoint parsed: file=C:\Users\user\Downloads\ACD_Accra_Ghana\Accra_2026\agentcore-samples\06-workshops\07-AgentCore-evaluations\00-prereqs\eval_agent_langgraph.py, bedrock_agentcore_name=eval_agent_langgraph
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: ac_eval_langgraph2


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

📄 Using existing Dockerfile: 
c:\Users\user\Downloads\ACD_Accra_Ghana\Accra_2026\agentcore-samples\06-workshops\07-AgentCore-evaluations\00-prere
qs\Dockerfile

Generated .dockerignore: c:\Users\user\Downloads\ACD_Accra_Ghana\Accra_2026\agentcore-samples\06-workshops\07-AgentCore-evaluations\00-prereqs\.dockerignore
Changing default agent from 'ac_eval_strands2' to 'ac_eval_langgraph2'
Bedrock AgentCore configured: c:\Users\user\Downloads\ACD_Accra_Ghana\Accra_2026\agentcore-samples\06-workshops\07-AgentCore-evaluations\00-prereqs\.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'ac_eval_langgraph2' to account 056187302207 (us-east-1)
Generated image tag: 20260826-062934-216
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository f

Repository doesn't exist, creating new ECR repository: bedrock-agentcore-ac_eval_langgraph2


ECR repository available: 056187302207.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-ac_eval_langgraph2
Getting or creating execution role for agent: ac_eval_langgraph2
Using AWS region: us-east-1, account ID: 056187302207
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-a18a56d054
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-a18a56d054
Starting execution role creation process for agent: ac_eval_langgraph2
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-east-1-a18a56d054
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-a18a56d054
✓ Role created: arn:aws:iam::056187302207:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-a18a56d054
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-ac_eval_langgraph2
Role creation complete and ready for use with Bedrock AgentCore
Execution role available: arn:aws:iam::056187302207:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-a18a56d054
Preparing CodeBuild project

LangGraph agent deployment started: mode='codebuild' tag='bedrock_agentcore-ac_eval_langgraph2:None' env_vars=None port=None runtime=None ecr_uri='056187302207.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-ac_eval_langgraph2:20260826-062934-216' agent_id='ac_eval_langgraph2-w499WCCilV' agent_arn='arn:aws:bedrock-agentcore:us-east-1:056187302207:runtime/ac_eval_langgraph2-w499WCCilV' codebuild_id='bedrock-agentcore-ac_eval_langgraph2-builder:c5d78c89-9153-491e-bee7-f2e90a79d05c' build_output=None


### Check the status of LangGraph agent
Now that we've deployed the LangGraph agent to AgentCore Runtime, let's check for it's deployment status

In [55]:
langgraph_status = wait_for_deployment(langgraph_agentcore_runtime, "LangGraph")

Retrieved Bedrock AgentCore status for: ac_eval_langgraph2


LangGraph status: READY
LangGraph deployment completed with status: READY


### Invoke the Langgraph Agent on Runtime
Test the LangGraph agent endpoint on AgentCore Runtime with a payload:

In [56]:
session_id_langgraph = str(uuid.uuid4())
print(f"Session ID: {session_id_langgraph}")

Session ID: 20d38c68-8361-4297-81ea-f52ebceb7efc


In [60]:
response = langgraph_agentcore_runtime.invoke(payload={"prompt": "What is 2+2?"}, session_id=session_id_langgraph)
print(response)

{'ResponseMetadata': {'RequestId': '2bef073b-3610-415f-83d8-c1c26b2b4ed5', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Wed, 26 Aug 2026 06:37:36 GMT', 'content-type': 'application/json', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'x-amzn-requestid': '2bef073b-3610-415f-83d8-c1c26b2b4ed5', 'x-amzn-bedrock-agentcore-runtime-session-id': '20d38c68-8361-4297-81ea-f52ebceb7efc'}, 'RetryAttempts': 0}, 'runtimeSessionId': '20d38c68-8361-4297-81ea-f52ebceb7efc', 'contentType': 'application/json', 'statusCode': 200, 'response': ['2 + 2 = **4**']}


In [58]:
invoke_response = agentcore_runtime.invoke(
    payload={"prompt": "What is the weather now?"}, session_id=session_id_langgraph
)
invoke_response

{'ResponseMetadata': {'RequestId': '6b62b840-00bd-4282-8a69-5434647e3154',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 26 Aug 2026 06:36:23 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': '6b62b840-00bd-4282-8a69-5434647e3154',
   'x-amzn-bedrock-agentcore-runtime-session-id': '20d38c68-8361-4297-81ea-f52ebceb7efc'},
  'RetryAttempts': 0},
 'runtimeSessionId': '20d38c68-8361-4297-81ea-f52ebceb7efc',
 'contentType': 'application/json',
 'statusCode': 200,
 'response': ['The weather right now is **sunny**! ☀️']}

In [61]:
invoke_response = agentcore_runtime.invoke(
    payload={"prompt": "Can you tell me the capital of the US?"},
    session_id=session_id_langgraph,
)
invoke_response

{'ResponseMetadata': {'RequestId': '70be94be-0b55-465a-ada5-cd8a70cf3f7d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 26 Aug 2026 06:37:49 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': '70be94be-0b55-465a-ada5-cd8a70cf3f7d',
   'x-amzn-bedrock-agentcore-runtime-session-id': '20d38c68-8361-4297-81ea-f52ebceb7efc'},
  'RetryAttempts': 0},
 'runtimeSessionId': '20d38c68-8361-4297-81ea-f52ebceb7efc',
 'contentType': 'application/json',
 'statusCode': 200,
 'response': ["The capital of the United States is **Washington, D.C.** (Washington, District of Columbia).\n\nWashington, D.C. is located on the eastern coast of the US and has been the capital since 1800. It's home to the White House (the President's residence), the Capitol Building (where Congress meets), and many other important government institutions and monuments."]}

In [62]:
print(f"Strands: {launch_result_strands}, Session: {session_id_strands}")
print(f"LangGraph: {launch_result_langgraph}, Session: {session_id_langgraph}")

Strands: mode='codebuild' tag='bedrock_agentcore-ac_eval_strands2:None' env_vars=None port=None runtime=None ecr_uri='056187302207.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-ac_eval_strands2:20260826-055953-054' agent_id='ac_eval_strands2-m5noRdAhvA' agent_arn='arn:aws:bedrock-agentcore:us-east-1:056187302207:runtime/ac_eval_strands2-m5noRdAhvA' codebuild_id='bedrock-agentcore-ac_eval_strands2-builder:7dda950a-5751-468f-a372-3bce38580c1e' build_output=None, Session: 49f812fe-a8f4-4891-8fa8-a1722672e3ef
LangGraph: mode='codebuild' tag='bedrock_agentcore-ac_eval_langgraph2:None' env_vars=None port=None runtime=None ecr_uri='056187302207.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-ac_eval_langgraph2:20260826-062934-216' agent_id='ac_eval_langgraph2-w499WCCilV' agent_arn='arn:aws:bedrock-agentcore:us-east-1:056187302207:runtime/ac_eval_langgraph2-w499WCCilV' codebuild_id='bedrock-agentcore-ac_eval_langgraph2-builder:c5d78c89-9153-491e-bee7-f2e90a79d05c' build_output=None, Sess

In [63]:
%store launch_result_strands
%store session_id_strands
%store launch_result_langgraph
%store session_id_langgraph

Stored 'launch_result_strands' (LaunchResult)
Stored 'session_id_strands' (str)
Stored 'launch_result_langgraph' (LaunchResult)
Stored 'session_id_langgraph' (str)


## Next Steps

Now that you have all the required pre-requisites, let's go through the individual evaluation tutorials:
Continue with the evaluation tutorials:
- [01-creating-custom-evaluators](../01-creating-custom-evaluators/): Create custom evaluators
- [02-running-evaluations](../02-running-evaluations/): Run on-demand and online evaluations
- [03-advanced](../03-advanced/): : Advanced techniques and dashboards